In [36]:
import json
import pandas as pd

# 1. Les data
with open("traffic.jsonl", encoding="utf-8") as file:
    df = pd.DataFrame(json.loads(line) for line in file if line.strip())

df["departure"] = pd.to_datetime(df["depature"], format="%H:%M")
df["arrival"] = pd.to_datetime(df["arrival"], format="%H:%M")

df["duration"] = (
    (df["arrival"] - df["departure"]).dt.total_seconds() / 60
)

df["departure_minutes"] = (
    df["departure"].dt.hour * 60
    + df["departure"].dt.minute
)

# 2. Del hver rute i trening og validering
train_dfs = {}
val_dfs = {}

for route, data in df.groupby("road"):
    train = data.sample(frac=0.8, random_state=42)
    val = data.drop(index=train.index)

    train_dfs[route] = train.copy()
    val_dfs[route] = val.copy()

# 3. Finn felles min/maks fra kun treningsdataene
all_train = pd.concat(train_dfs.values())

x_min = all_train["departure_minutes"].min()
x_max = all_train["departure_minutes"].max()
x_range = x_max - x_min

print(f"Min avgangstid: {x_min} min, maks avgangstid: {x_max} min")

if x_range == 0:
    raise ValueError("Treningsdataene må ha ulike avgangstider.")

# 4. Skaler begge settene med treningens grenser
for route in train_dfs:
    for data in (train_dfs[route], val_dfs[route]):
        data["x"] = (data["departure_minutes"] - x_min) / x_range
        data["y"] = data["duration"]

# Vis antall målinger per rute
pd.DataFrame({
    "Trening": {route: len(data) for route, data in train_dfs.items()},
    "Validering": {route: len(data) for route, data in val_dfs.items()},
})

Min avgangstid: 420 min, maks avgangstid: 1019 min


,Trening,Validering
A->C->D,202,50
A->C->E,210,53
B->C->D,207,52
B->C->E,206,51


In [37]:
import plotly.express as px

for route, data in train_dfs.items():
    fig = px.scatter(
        data,
        x="x",
        y="y",
        title=f"Treningsdata: {route}",
        labels={
            "x": "Skalert avgangstid",
            "y": "Reisetid (minutter)"
        },
        hover_data=["departure_minutes"]
    )

    fig.show(renderer="vscode")

In [38]:
import numpy as np

def sample_theta_acd(size_of_theta):
    return np.random.uniform(
        low=[0, 0, 0, 60],
        high=[60, 4 * np.pi, 2 * np.pi, 140],
        size=size_of_theta
    )

In [39]:
def get_loss(y_hat, ys):
    # No change needed, returns quadratic loss.
    loss = ((y_hat - ys)**2).sum()
    return loss

In [40]:
def pred_acd(x, theta):
    a, b, c, d = theta
    return a * np.sin(b * x + c) + d

In [30]:
import tqdm 

data_acd = train_dfs["A->C->D"]
xs = data_acd["x"].to_numpy()
ys = data_acd["y"].to_numpy()
n_params = 4

best_theta = sample_theta_acd(n_params)
best_loss = float('inf')

for i in tqdm.tqdm(range(1000000)):
    if i < 10000 or np.random.random() < 0.20:
        curr_theta = sample_theta_acd(n_params)
    else:
        curr_theta = best_theta.copy()
        index = np.random.randint(n_params)

        sampled_theta = sample_theta_acd(n_params)
        curr_theta[index] = sampled_theta[index]

    y_hat = pred_acd(xs, curr_theta)
    curr_loss = get_loss(y_hat, ys)

    if curr_loss < best_loss:
        best_loss = curr_loss
        best_theta = curr_theta

print(f"Best loss: {best_loss}")
print(f"Best theta: {best_theta}")

x_plot = np.linspace(xs.min(), xs.max(), 500)

fig = px.line(
    x=x_plot,
    y=pred_acd(x_plot, best_theta),
    title="Prediksjon med beste theta",
    labels={"x": "Skalert avgangstid", "y": "Reisetid (minutter)"}
)
fig.add_scatter(x=xs, y=ys, mode="markers", name="Treningsdata")
fig.show(renderer="vscode")

rmse = np.sqrt(best_loss / len(ys))
print(f"RMSE på treningsdata: {rmse:.2f} minutter")



100%|██████████| 1000000/1000000 [00:16<00:00, 60710.71it/s]

Best loss: 5835.0805697043415
Best theta: [ 33.52533001   4.79692269   2.31996204 105.43875151]


RMSE på treningsdata: 5.37 minutter


In [31]:
val_acd = val_dfs["A->C->D"]

xs_val = val_acd["x"].to_numpy()
ys_val = val_acd["y"].to_numpy()

y_hat_val = pred_acd(xs_val, best_theta)
rmse_val = np.sqrt(np.mean((y_hat_val - ys_val) ** 2))

print(f"RMSE trening: {np.sqrt(best_loss / len(ys)):.2f} minutter")
print(f"RMSE validering: {rmse_val:.2f} minutter")

RMSE trening: 5.37 minutter
RMSE validering: 5.36 minutter


In [41]:
def sample_theta_ace():
    a = np.random.uniform(0, 20)
    b = np.random.uniform(90, 105)
    return np.array([a, b])


In [42]:

def pred_ace(x, theta):
    a, b = theta
    return a * x + b
import tqdm 


In [12]:


data_ace = train_dfs["A->C->E"]
xs = data_ace["x"].to_numpy()
ys = data_ace["y"].to_numpy()
n_params = 2

best_theta = sample_theta_ace()
best_loss = float('inf')

for i in tqdm.tqdm(range(1000000)):
    if i < 10000 or np.random.random() < 0.20:
        curr_theta = sample_theta_ace()
    else:
        curr_theta = best_theta.copy()
        index = np.random.randint(n_params)

        sampled_theta = sample_theta_ace()
        curr_theta[index] = sampled_theta[index]

    y_hat = pred_ace(xs, curr_theta)
    curr_loss = get_loss(y_hat, ys)

    if curr_loss < best_loss:
        best_loss = curr_loss
        best_theta = curr_theta

print(f"Best loss: {best_loss}")
print(f"Best theta: {best_theta}")

x_plot = np.linspace(xs.min(), xs.max(), 500)

fig = px.line(
    x=x_plot,
    y=pred_ace(x_plot, best_theta),
    title="Prediksjon med beste theta",
    labels={"x": "Skalert avgangstid", "y": "Reisetid (minutter)"}
)
fig.add_scatter(x=xs, y=ys, mode="markers", name="Treningsdata")
fig.show(renderer="vscode")

rmse = np.sqrt(best_loss / len(ys))
print(f"RMSE på treningsdata: {rmse:.2f} minutter")



  0%|          | 0/1000000 [00:00<?, ?it/s]

100%|██████████| 1000000/1000000 [00:16<00:00, 61741.10it/s]


Best loss: 4322.494166195618
Best theta: [ 1.07686462 97.16703124]


RMSE på treningsdata: 4.54 minutter


#BCD Model

In [43]:
def sample_theta_bcd():
    return np.random.uniform(
        low=[80, 0, 0, 0, 385, -0.1, 0.08],
        high=[180, 120, 4 *np.pi, 2 * np.pi, 625, 0.1 , 0.13],
        size= 7
    )

In [44]:
def get_loss(y_hat, ys):
    # No change needed, returns quadratic loss.
    loss = ((y_hat - ys)**2).sum()
    return loss

In [45]:
def pred_bcd(x, theta):
    a, b, c, d, e, f, g = theta
    return a + b * np.sin(c * x + d) - e*((x-f)% g)

In [23]:
data_bcd = train_dfs["B->C->D"]
xs = data_bcd["x"].to_numpy()
ys = data_bcd["y"].to_numpy()
n_params = 7

best_theta = sample_theta_bcd()
best_loss = float('inf')

for i in tqdm.tqdm(range(1000000)):
    if i < 10000 or np.random.random() < 0.20:
        curr_theta = sample_theta_bcd()
    else:
        curr_theta = best_theta.copy()
        index = np.random.randint(n_params)

        sampled_theta = sample_theta_bcd()
        curr_theta[index] = sampled_theta[index]

    y_hat = pred_bcd(xs, curr_theta)
    curr_loss = get_loss(y_hat, ys)

    if curr_loss < best_loss:
        best_loss = curr_loss
        best_theta = curr_theta

print(f"Best loss: {best_loss}")
print(f"Best theta: {best_theta}")

x_plot = np.linspace(xs.min(), xs.max(), 500)

fig = px.line(
    x=x_plot,
    y=pred_bcd(x_plot, best_theta),
    title="Prediksjon med beste theta",
    labels={"x": "Skalert avgangstid", "y": "Reisetid (minutter)"}
)
fig.add_scatter(x=xs, y=ys, mode="markers", name="Treningsdata")
fig.show(renderer="vscode")

rmse = np.sqrt(best_loss / len(ys))
print(f"RMSE på treningsdata: {rmse:.2f} minutter")

100%|██████████| 1000000/1000000 [00:35<00:00, 28399.34it/s]

Best loss: 4151.6382988174855
Best theta: [1.40993318e+02 3.26487772e+01 4.79431594e+00 2.33387954e+00
 6.11884394e+02 2.83217492e-02 9.97641125e-02]


RMSE på treningsdata: 4.48 minutter


In [24]:
val_bcd = val_dfs["B->C->D"]

xs_val = val_bcd["x"].to_numpy()
ys_val = val_bcd["y"].to_numpy()

y_hat_val = pred_bcd(xs_val, best_theta)
rmse_val = np.sqrt(np.mean((y_hat_val - ys_val) ** 2))

print(f"RMSE trening: {np.sqrt(best_loss / len(ys)):.2f} minutter")
print(f"RMSE validering: {rmse_val:.2f} minutter")

RMSE trening: 4.48 minutter
RMSE validering: 4.07 minutter


In [46]:
def sample_theta_bce():
    return np.random.uniform(
        low=[120, 420, -0.2, 0.08],
        high=[150, 787, 0.2, 0.15],
        size=4
    )

In [47]:
def pred_bce(x, theta):
    a, b, c, d = theta
    return a - b * ((x - c) % d)

In [33]:
data_bce = train_dfs["B->C->E"]
xs = data_bce["x"].to_numpy()
ys = data_bce["y"].to_numpy()
n_params = 4

best_theta = sample_theta_bce()
best_loss = float('inf')

for i in tqdm.tqdm(range(1000000)):
    if i < 10000 or np.random.random() < 0.20:
        curr_theta = sample_theta_bce()
    else:
        curr_theta = best_theta.copy()
        index = np.random.randint(n_params)

        sampled_theta = sample_theta_bce()
        curr_theta[index] = sampled_theta[index]

    y_hat = pred_bce(xs, curr_theta)
    curr_loss = get_loss(y_hat, ys)

    if curr_loss < best_loss:
        best_loss = curr_loss
        best_theta = curr_theta

print(f"Best loss: {best_loss}")
print(f"Best theta: {best_theta}")

x_plot = np.linspace(xs.min(), xs.max(), 500)

fig = px.line(
    x=x_plot,
    y=pred_bce(x_plot, best_theta),
    title="Prediksjon med beste theta",
    labels={"x": "Skalert avgangstid", "y": "Reisetid (minutter)"}
)
fig.add_scatter(x=xs, y=ys, mode="markers", name="Treningsdata")
fig.show(renderer="vscode")

rmse = np.sqrt(best_loss / len(ys))
print(f"RMSE på treningsdata: {rmse:.2f} minutter")

100%|██████████| 1000000/1000000 [00:25<00:00, 39376.27it/s]

Best loss: 1688.6733786888549
Best theta: [1.32146719e+02 5.92088606e+02 1.26169775e-01 1.00159422e-01]


RMSE på treningsdata: 2.86 minutter


In [34]:
val_bce = val_dfs["B->C->E"]

xs_val = val_bce["x"].to_numpy()
ys_val = val_bce["y"].to_numpy()

y_hat_val = pred_bce(xs_val, best_theta)
rmse_val = np.sqrt(np.mean((y_hat_val - ys_val) ** 2))

print(f"RMSE trening: {np.sqrt(best_loss / len(ys)):.2f} minutter")
print(f"RMSE validering: {rmse_val:.2f} minutter")

RMSE trening: 2.86 minutter
RMSE validering: 3.40 minutter


In [48]:
import numpy as np
import plotly.graph_objects as go

# Parameterne som ble brukt i bildet.
models = {
    "B->C->E": (
        pred_bce,
        [132.146719, 592.088606, 0.126169775, 0.100159422],
        "#636EFA"
    ),
    "A->C->E": (
        pred_ace,
        [1.07686462, 97.16703124],
        "#EF553B"
    ),
    "B->C->D": (
        pred_bcd,
        [
            140.993318, 32.6487772, 4.79431594, 2.33387954,
            611.884394, 0.0283217492, 0.0997641125
        ],
        "#00CC96"
    ),
    "A->C->D": (
        pred_acd,
        [33.52533001, 4.79692269, 2.31996204, 105.43875151],
        "#AB63FA"
    )
}

# Min–maks-grensene fra treningen som ga disse parameterne.
train_min = 420
train_max = 1019

departure_plot = np.linspace(train_min, train_max, 10000)
x_scaled = (departure_plot - train_min) / (train_max - train_min)

fig = go.Figure()

# Alle datapunktene, inkludert valideringsdata.
for route, (_, _, color) in models.items():
    data = df[df["road"] == route]

    fig.add_trace(go.Scatter(
        x=data["departure_minutes"],
        y=data["duration"],
        customdata=data["departure"].dt.strftime("%H:%M"),
        mode="markers",
        marker=dict(color=color, size=6, opacity=0.4),
        name=route,
        legendgroup=route,
        showlegend=False,
        hovertemplate=(
            "Avgang: %{customdata}<br>"
            "Reisetid: %{y:.1f} min<extra>%{fullData.name}</extra>"
        )
    ))

# Modellene får skalert input; aksen viser minutter/klokkeslett.
for route, (predict, theta, color) in models.items():
    fig.add_trace(go.Scatter(
        x=departure_plot,
        y=predict(x_scaled, np.array(theta)),
        mode="lines",
        line=dict(color=color, width=3),
        name=route,
        legendgroup=route,
        hovertemplate=(
            "Predikert reisetid: %{y:.1f} min"
            "<extra>%{fullData.name}</extra>"
        )
    ))

fig.update_layout(
    title="Alle målinger og de fire modellene",
    template="plotly_white",
    xaxis_title="Avgangstid",
    yaxis_title="Reisetid (minutter)",
    height=650,
    legend=dict(
        orientation="h",
        x=0.5,
        xanchor="center",
        y=-0.2,
        groupclick="togglegroup"
    )
)

fig.update_xaxes(
    tickvals=[h * 60 for h in range(7, 18)],
    ticktext=[f"{h:02d}:00" for h in range(7, 18)],
    range=[420, 1020]
)

fig.show(renderer="vscode")